In [1]:
# ============================================================
# COLAB USERS ONLY — Uncomment and run this cell if using Google Colab
# If you're running locally in VS Code, SKIP this cell entirely
# ============================================================

!pip install langchain-groq langchain-google-genai langchain-community python-dotenv -q

from google.colab import userdata
import os
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
print("Colab setup complete!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 26.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
Colab setup complete!


# Day 3: Agent Memory Systems 🧠

**Agentic AI Hands-On Course** | Dr. Kanthi Kiran Sirra | Sr. AI Engineer

**What you'll build today:**
1. Prove that LLMs have zero memory — with a live demo
2. Build buffer memory from scratch using plain Python dicts
3. Build sliding window memory from scratch
4. Use LangChain's memory classes (Buffer, Window, Summary, Entity)
5. Build a chatbot that lets you switch memory types on the fly
6. Extend the AgentLLM class from Day 1 with memory support

**Time:** 1.5 hours | **Difficulty:** Beginner-friendly | **Prerequisites:** Day 1 + Day 2 complete

**Framework:** LangChain (memory) | **LLM:** Groq API

---

## 0. Setup — Load your API keys

Run this cell first. It loads your keys from the `.env` file.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_key = os.getenv("GROQ_API_KEY", "")
gemini_key = os.getenv("GOOGLE_API_KEY", "")

print(f"Groq API Key:    {'✅ Loaded' if len(groq_key) > 10 else '❌ Missing — check .env file'}")
print(f"Gemini API Key:  {'✅ Loaded' if len(gemini_key) > 10 else '⚠️  Optional — not critical for today'}")

Groq API Key:    ✅ Loaded
Gemini API Key:  ✅ Loaded


---
## 1. 🎯 The Problem — Prove That LLMs Have Zero Memory

Before building anything, let's see the problem with our own eyes.

Every time you call `llm.invoke()`, it's a completely fresh HTTP request. The model has no memory of previous calls.

Watch what happens when we make 3 separate calls:

In [3]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Three completely separate invoke() calls
r1 = llm.invoke("My name is Ravi and I am from Hyderabad.")
print(f"Turn 1: {r1.content}")
print()

r2 = llm.invoke("I love Python programming.")
print(f"Turn 2: {r2.content}")
print()

r3 = llm.invoke("What is my name and where am I from?")
print(f"Turn 3: {r3.content}")
print()
print("❌ The AI has no idea! Each invoke() is a fresh start.")

Turn 1: Nice to meet you, Ravi. Hyderabad is a beautiful city with a rich history and culture. I hope you're doing well. Is there something I can help you with or would you like to chat about your city or anything else that's on your mind?

Turn 2: Python is a fantastic language to work with, known for its simplicity, readability, and versatility. It has a wide range of applications, from web development and data analysis to artificial intelligence and machine learning.

What do you like most about Python programming? Is it the ease of learning, the vast number of libraries and frameworks, or something else? Are you working on any exciting projects or exploring specific areas like data science, automation, or game development?

Turn 3: I don't have any information about your name or where you're from. I'm a large language model, I don't have the ability to access personal information about individuals, and our conversation just started, so I don't have any prior knowledge about you. If

**The LLM doesn't know your name because each call is completely isolated.**

This is the core problem memory systems solve. The LLM itself will never have memory — it's physically impossible given how inference works. Memory is always an **application-layer engineering decision** that you make.

---
## 2. Memory from Scratch — Python Dicts (No Frameworks)

Before using LangChain, let's build memory manually. This is exactly what all frameworks do internally.

### 2.1 Buffer Memory — Store Everything

In [4]:
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# ── Buffer Memory from scratch ──────────────────────────────
# memory lives OUTSIDE the function so it persists between calls
memory = []

def chat_with_buffer_memory(user_input, system_prompt="You are a helpful assistant. Be concise."):
    """Chat function with full buffer memory."""
    # Step 1: add system prompt on first turn only
    if len(memory) == 0:
        memory.append(SystemMessage(content=system_prompt))

    # Step 2: write user message into memory
    memory.append(HumanMessage(content=user_input))

    # Step 3: send ENTIRE memory list to LLM
    response = llm.invoke(memory).content

    # Step 4: write AI reply into memory
    memory.append(AIMessage(content=response))

    return response


# Test — same 3 questions as before
print("Turn 1:", chat_with_buffer_memory("My name is Ravi and I am from Hyderabad."))
print()
print("Turn 2:", chat_with_buffer_memory("I love Python programming."))
print()
print("Turn 3:", chat_with_buffer_memory("What is my name and where am I from?"))
print()
print(f"✅ Memory now holds {len(memory)} messages (1 system + {len(memory)-1} conversation turns)")

Turn 1: Hello Ravi, nice to meet you. How can I assist you today?

Turn 2: Python is a great language. What do you like to do with Python, Ravi - web development, data analysis, or something else?

Turn 3: Your name is Ravi, and you're from Hyderabad.

✅ Memory now holds 7 messages (1 system + 6 conversation turns)


In [5]:
# Let's inspect what's inside memory — this is what gets sent to the LLM each time
print("=== Contents of memory list ===")
for i, msg in enumerate(memory):
    role = type(msg).__name__.replace('Message', '')
    content_preview = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
    print(f"  [{i}] {role:10s} → {content_preview}")

print()
print("⚠️  Problem: This list grows forever. Turn 100 would send 200 messages to the LLM!")

=== Contents of memory list ===
  [0] System     → You are a helpful assistant. Be concise.
  [1] Human      → My name is Ravi and I am from Hyderabad.
  [2] AI         → Hello Ravi, nice to meet you. How can I assist you today?
  [3] Human      → I love Python programming.
  [4] AI         → Python is a great language. What do you like to do with Pyth...
  [5] Human      → What is my name and where am I from?
  [6] AI         → Your name is Ravi, and you're from Hyderabad.

⚠️  Problem: This list grows forever. Turn 100 would send 200 messages to the LLM!


### 2.2 Sliding Window Memory — Cap the Cost

Simple fix: only send the last N messages. Old messages get dropped.

In [6]:
# Full memory history — keeps everything for reference
full_memory = []

def chat_with_window_memory(user_input, window=4, system_prompt="You are a helpful assistant. Be concise."):
    """Chat function that only sends the last `window` messages to the LLM."""
    # Add to full history (we never discard from full_memory)
    full_memory.append(HumanMessage(content=user_input))

    # Build what we ACTUALLY send: system + last `window` messages
    system = [SystemMessage(content=system_prompt)]
    recent = full_memory[-window:]   # ← only last N messages
    to_send = system + recent

    response = llm.invoke(to_send).content
    full_memory.append(AIMessage(content=response))

    print(f"  [Sent {len(to_send)} messages to LLM | Full history: {len(full_memory)} messages]")
    return response


print("=== Sliding Window Memory (window=4) ===")
print()
print("Turn 1:", chat_with_window_memory("My name is Ravi."))
print("Turn 2:", chat_with_window_memory("I am from Hyderabad."))
print("Turn 3:", chat_with_window_memory("I love Python."))
print("Turn 4:", chat_with_window_memory("I am learning LangChain."))
print()
# At this point window=4 means turns 1-4 are all still in window
print("Turn 5:", chat_with_window_memory("What is my name?"))  # Still remembers
print()
print("Turn 6:", chat_with_window_memory("What city am I from?"))
# Turn 1 ('My name is Ravi') has now been pushed out of the window!
print()
print("Turn 7:", chat_with_window_memory("What is my name?"))  # ← may forget now!

=== Sliding Window Memory (window=4) ===

  [Sent 2 messages to LLM | Full history: 2 messages]
Turn 1: Hello Ravi, how can I help you?
  [Sent 4 messages to LLM | Full history: 4 messages]
Turn 2: Nice to know that, Ravi. Hyderabad is a beautiful city. What's on your mind?
  [Sent 5 messages to LLM | Full history: 6 messages]
Turn 3: Python is a great language. What do you want to know or do with Python?
  [Sent 5 messages to LLM | Full history: 8 messages]
Turn 4: LangChain is a powerful framework for building AI applications. What specific aspect of LangChain are you struggling with or would like to learn more about?

  [Sent 5 messages to LLM | Full history: 10 messages]
Turn 5: I don't know your name. You haven't shared it with me. Would you like to introduce yourself?

  [Sent 5 messages to LLM | Full history: 12 messages]
Turn 6: I don't have that information. Our conversation just started, and you haven't shared any details about your location.

  [Sent 5 messages to LLM | Full

---
## ✅ Quick Check — What will happen?

Before running the next cell, predict: if `window=2`, and we ask in turn 5 "What is my name?" (told in turn 1) — will the AI remember?

**Think:** window=2 → only last 2 messages sent → turn 1 is long gone.

In [7]:
# Verify your prediction
small_memory = []

def chat_tiny_window(user_input, window=2):
    small_memory.append(HumanMessage(content=user_input))
    system = [SystemMessage(content="You are a helpful assistant.")]
    recent = small_memory[-window:]
    response = llm.invoke(system + recent).content
    small_memory.append(AIMessage(content=response))
    return response

chat_tiny_window("My name is Ravi.")           # Turn 1
chat_tiny_window("I love cricket.")             # Turn 2
chat_tiny_window("Python is my favourite.")     # Turn 3
chat_tiny_window("Today is a great day.")       # Turn 4
answer = chat_tiny_window("What is my name?")   # Turn 5
print(f"AI says: {answer}")
print()
print("Was your prediction correct? The window dropped turn 1 (my name) long ago!")

AI says: I don't have any information about your name. I'm a large language model, I don't have the ability to know your personal details or retain information about our previous conversations. Each time you interact with me, it's a new conversation. If you'd like to share your name with me, I'd be happy to address you by it!

Was your prediction correct? The window dropped turn 1 (my name) long ago!


---
## 3. LangChain Memory Classes — The Framework Way

LangChain provides pre-built memory classes that automate everything we just built manually.

The key class is `ConversationChain` — it wraps your LLM + memory into one object that handles the full loop automatically.

### 3.1 ConversationBufferMemory — The Baseline

In [8]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# ── Modern LangChain pattern ────────────────────────────────
# Step 1: Build a prompt template with a "history" placeholder
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Be concise."),
    MessagesPlaceholder(variable_name="history"),  # ← memory goes here
    ("human", "{input}"),
])

# Step 2: Chain prompt → LLM using the pipe operator
chain = prompt | llm

# Step 3: A store that holds chat history per session
store = {}
def get_history(session_id: str) -> ChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Step 4: Wrap the chain with history management
chain_with_history = RunnableWithMessageHistory(
    chain,
    get_history,
    input_messages_key="input",
    history_messages_key="history",
)

# ── Use it — config tells it which session to use ──────────
SESSION = {"configurable": {"session_id": "ravi_session"}}

print("=== Buffer Memory (Modern LangChain) ===")
print()

r1 = chain_with_history.invoke(
    {"input": "My name is Ravi and I work as a Data Scientist in Hyderabad."},
    config=SESSION
)
print(f"Turn 1: {r1.content}")
print()

r2 = chain_with_history.invoke(
    {"input": "I am currently learning Agentic AI frameworks."},
    config=SESSION
)
print(f"Turn 2: {r2.content}")
print()

r3 = chain_with_history.invoke(
    {"input": "Based on everything I told you, suggest one learning goal for me."},
    config=SESSION
)
print(f"Turn 3: {r3.content}")



=== Buffer Memory (Modern LangChain) ===

Turn 1: Hello Ravi, nice to meet you. What can I help you with today?

Turn 2: Agentic AI is a fascinating field. What specific aspects of Agentic AI frameworks are you struggling with or would like to know more about?

Turn 3: Based on your background as a Data Scientist and interest in Agentic AI, a potential learning goal for you could be: "Apply Agentic AI frameworks to real-world problems in data science, such as autonomous decision-making or multi-agent systems, within the next 3 months."


In [9]:
# Inspect what's stored in the history object
history = get_history("ravi_session")
print("=== Contents of memory ===")
for i, msg in enumerate(history.messages):
    role = type(msg).__name__.replace('Message', '')
    content_preview = msg.content[:60] + "..." if len(msg.content) > 60 else msg.content
    print(f"  [{i}] {role:10s} → {content_preview}")

print()
print(f"Total messages stored: {len(history.messages)}")
print("⚠️  Problem: This list grows forever. Turn 100 would send 200 messages to the LLM!")



=== Contents of memory ===
  [0] Human      → My name is Ravi and I work as a Data Scientist in Hyderabad.
  [1] AI         → Hello Ravi, nice to meet you. What can I help you with today...
  [2] Human      → I am currently learning Agentic AI frameworks.
  [3] AI         → Agentic AI is a fascinating field. What specific aspects of ...
  [4] Human      → Based on everything I told you, suggest one learning goal fo...
  [5] AI         → Based on your background as a Data Scientist and interest in...

Total messages stored: 6
⚠️  Problem: This list grows forever. Turn 100 would send 200 messages to the LLM!


### 3.2 ConversationBufferWindowMemory — Sliding Window, LangChain Style

In [11]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langchain_core.runnables.history import RunnableWithMessageHistory
from typing import List

# ── Window Memory — plain Python approach ──────────────────
# Simpler and cleaner than subclassing a Pydantic model

class WindowChatHistory:
    """Stores full history but only returns last k turns to the LLM."""

    def __init__(self, k: int = 5):
        self.k = k
        self._messages = []   # stores everything

    @property
    def messages(self) -> List[BaseMessage]:
        # Return only last k*2 messages (k turns = k human + k AI)
        cutoff = self.k * 2
        return self._messages[-cutoff:] if len(self._messages) > cutoff else self._messages

    def add_user_message(self, content: str):
        self._messages.append(HumanMessage(content=content))

    def add_ai_message(self, content: str):
        self._messages.append(AIMessage(content=content))


# ── Test it ─────────────────────────────────────────────────
from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
SYSTEM = SystemMessage(content="You are a helpful assistant. Be concise.")

win_history = WindowChatHistory(k=3)

print("=== Window Memory (k=3) ===")
print()

conversations = [
    "My name is Ravi.",
    "I am from Hyderabad.",
    "I love Python.",
    "I am learning LangChain.",      # Turn 4 — turn 1 should now be dropped
    "What is my name?",              # Turn 5 — will it remember?
]

for i, msg in enumerate(conversations, 1):
    win_history.add_user_message(msg)
    response = llm.invoke([SYSTEM] + win_history.messages).content
    win_history.add_ai_message(response)
    print(f"Turn {i} | Input: '{msg}'")
    print(f"        | Response: {response[:80]}..." if len(response) > 80 else f"        | Response: {response}")
    print(f"        | Messages visible to LLM: {len(win_history.messages)}")
    print()

=== Window Memory (k=3) ===

Turn 1 | Input: 'My name is Ravi.'
        | Response: Hello Ravi, how can I help you today?
        | Messages visible to LLM: 2

Turn 2 | Input: 'I am from Hyderabad.'
        | Response: Hyderabad is a beautiful city. What would you like to know or discuss, Ravi?
        | Messages visible to LLM: 4

Turn 3 | Input: 'I love Python.'
        | Response: Python is a great language. Do you have a specific project or topic in Python yo...
        | Messages visible to LLM: 6

Turn 4 | Input: 'I am learning LangChain.'
        | Response: LangChain is an exciting framework for building AI applications. What specific a...
        | Messages visible to LLM: 6

Turn 5 | Input: 'What is my name?'
        | Response: Your name is Ravi.
        | Messages visible to LLM: 6



### 3.3 ConversationSummaryMemory — Compress the Past

This is where it gets interesting. Instead of dropping old messages, the LLM writes a running summary of the conversation so far.

In [12]:
# ── Summary Memory — manual implementation ─────────────────
# LangChain 0.3+ removed ConversationSummaryMemory.
# We implement the same logic manually — which is actually MORE educational
# because you see exactly how it works under the hood.

from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

class SummaryMemory:
    """
    Summary memory: keeps a running LLM-generated summary of old turns,
    plus the last few raw turns. Exactly what LangChain used to do internally.
    """
    def __init__(self, llm, keep_last: int = 4):
        self.llm       = llm
        self.keep_last = keep_last   # how many recent messages to keep raw
        self.summary   = ""          # compressed history of old turns
        self.recent    = []          # last N messages kept verbatim

    def add(self, role: str, content: str):
        """Add a message and compress if needed."""
        msg = HumanMessage(content=content) if role == "user" else AIMessage(content=content)
        self.recent.append(msg)
        # When recent grows too large, compress the oldest messages
        if len(self.recent) > self.keep_last:
            self._compress()

    def _compress(self):
        """Ask the LLM to summarize the oldest messages."""
        to_compress = self.recent[:-self.keep_last]   # everything except last N
        self.recent  = self.recent[-self.keep_last:]  # keep only last N
        lines = []
        for m in to_compress:
            role = "Human" if isinstance(m, HumanMessage) else "AI"
            lines.append(f"{role}: {m.content}")
        compress_prompt = "Summarize this conversation in 2-3 sentences:\n\n" + "\n".join(lines)
        if self.summary:
            compress_prompt += f"\n\nPrevious summary: {self.summary}"
        self.summary = self.llm.invoke(compress_prompt).content
        print(f"  [Summary updated: {len(self.summary)} chars]")

    def get_messages(self):
        """Build the message list to send to the LLM."""
        msgs = [SystemMessage(content="You are a helpful assistant. Be concise.")]
        if self.summary:
            msgs.append(SystemMessage(content=f"Summary of earlier conversation: {self.summary}"))
        msgs.extend(self.recent)
        return msgs

    def chat(self, user_input: str) -> str:
        """Send a message and get a response."""
        self.add("user", user_input)
        response = self.llm.invoke(self.get_messages()).content
        self.add("ai", response)
        return response


# ── Test it ─────────────────────────────────────────────────
summary_mem = SummaryMemory(llm=llm, keep_last=4)

print("=== Summary Memory ===")
print()

turns = [
    "My name is Ravi. I work as a Data Scientist at a fintech company in Hyderabad.",
    "I have 3 years of experience in Python and Machine Learning.",
    "I am currently enrolled in an Agentic AI course to upskill.",
    "My goal is to build production-grade AI agents by the end of this course.",
    "What do you know about me so far?",
]

for i, msg in enumerate(turns, 1):
    response = summary_mem.chat(msg)
    print(f"Turn {i}: {response[:100]}..." if len(response) > 100 else f"Turn {i}: {response}")
    print()

print("=" * 50)
print("What the summary memory holds:")
print(f"Summary: {summary_mem.summary}")
print(f"Recent messages: {len(summary_mem.recent)}")
print()
print("Note: The LLM was called EXTRA times to generate the summary!")



=== Summary Memory ===

Turn 1: Hello Ravi, nice to meet you. What can I help you with today?

Turn 2: That's great experience to have, Ravi. How can I assist you with your Python or Machine Learning ski...

  [Summary updated: 215 chars]
  [Summary updated: 280 chars]
Turn 3: Upskilling is a great move. Agentic AI is an advanced topic. What specific areas of Agentic AI are y...

  [Summary updated: 262 chars]
  [Summary updated: 305 chars]
Turn 4: Ambitious goal. Building production-grade AI agents requires a deep understanding of the concepts an...

  [Summary updated: 327 chars]
  [Summary updated: 460 chars]
Turn 5: You're currently enrolled in an Agentic AI course, have a background in Python and Machine Learning,...

What the summary memory holds:
Summary: This conversation has just begun, with the human sharing their enrollment in an Agentic AI course to upskill, building on their 3-year background in Python and Machine Learning. The AI has expressed enthusiasm for the human's u

### 3.4 ConversationEntityMemory — Track Named Things

Entity memory doesn't summarize everything — it specifically tracks **facts about named entities** (people, places, projects, companies).

In [13]:
# ── Entity Memory — manual implementation ──────────────────
import json as json_lib
import re

class EntityMemory:
    """
    Entity memory: extracts and tracks facts about named entities
    (people, places, projects, companies) using the LLM.
    Same logic as LangChain's ConversationEntityMemory — implemented manually.
    """
    def __init__(self, llm):
        self.llm      = llm
        self.entities = {}   # {"Ravi": "Data Scientist from Hyderabad", ...}
        self.history  = []   # full message history

    def _extract_entities(self, text: str):
        """Ask LLM to extract entities from a piece of text."""
        extract_prompt = f"""Extract named entities (people, places, projects, companies) and one-line facts about them from this text.
Return ONLY a JSON object like: {{"EntityName": "fact about entity"}}
If no entities found, return {{}}
Text: {text}"""
        try:
            raw = self.llm.invoke(extract_prompt).content
            match = re.search(r'\{.*\}', raw, re.DOTALL)
            if match:
                new_entities = json_lib.loads(match.group())
                for name, fact in new_entities.items():
                    if name in self.entities:
                        self.entities[name] += f". {fact}"
                    else:
                        self.entities[name] = fact
        except Exception:
            pass  # graceful — don't crash if extraction fails

    def _build_entity_context(self, user_input: str) -> str:
        """Find which stored entities are relevant to the current message."""
        relevant = []
        for name, fact in self.entities.items():
            if name.lower() in user_input.lower():
                relevant.append(f"{name}: {fact}")
        return "\n".join(relevant) if relevant else ""

    def chat(self, user_input: str) -> str:
        """Send a message, extract entities, get response."""
        # Step 1: extract entities from what the user just said
        self._extract_entities(user_input)

        # Step 2: build context from relevant stored entities
        entity_context = self._build_entity_context(user_input)

        # Step 3: build the prompt
        messages = [SystemMessage(content="You are a helpful assistant. Be concise.")]
        if entity_context:
            messages.append(SystemMessage(content=f"Known facts:\n{entity_context}"))
        for role, content in self.history:
            messages.append(HumanMessage(content=content) if role=="user" else AIMessage(content=content))
        messages.append(HumanMessage(content=user_input))

        response = self.llm.invoke(messages).content
        self.history.append(("user", user_input))
        self.history.append(("ai",   response))
        return response


# ── Test it ─────────────────────────────────────────────────
entity_mem = EntityMemory(llm=llm)

print("=== Entity Memory ===")
print()

turns = [
    "My name is Ravi and I work at DataCorp in Hyderabad.",
    "My colleague Priya is leading our RAG project called ProjectX.",
    "ProjectX uses LangChain and ChromaDB for document search.",
    "What can you tell me about ProjectX?",
    "What do you know about Priya?",
]

for i, msg in enumerate(turns, 1):
    response = entity_mem.chat(msg)
    print(f"Turn {i} | '{msg}'")
    print(f"         Response: {response[:100]}..." if len(response) > 100 else f"         Response: {response}")
    print()

print("=" * 50)
print("Entity store — what was extracted and tracked:")
for entity, facts in entity_mem.entities.items():
    print(f"  📌 {entity}: {facts}")



=== Entity Memory ===

Turn 1 | 'My name is Ravi and I work at DataCorp in Hyderabad.'
         Response: That's consistent with what I know: you're Ravi, an employee of DataCorp, which is located in Hydera...

Turn 2 | 'My colleague Priya is leading our RAG project called ProjectX.'
         Response: That matches the information I have: Priya is indeed leading ProjectX, which is a RAG project.

Turn 3 | 'ProjectX uses LangChain and ChromaDB for document search.'
         Response: That's correct. As I know, ProjectX utilizes both LangChain and ChromaDB for its document search fun...

Turn 4 | 'What can you tell me about ProjectX?'
         Response: ProjectX is a RAG project led by Priya, and it uses LangChain and ChromaDB for document search. That...

Turn 5 | 'What do you know about Priya?'
         Response: I know that Priya is leading ProjectX, a RAG project at DataCorp where you work, Ravi.

Entity store — what was extracted and tracked:
  📌 Ravi: works at DataCorp
  📌 DataCorp

---
## 4. Side-by-Side Comparison — The Forgetful vs Memory-Enabled Demo

Let's run the exact same conversation through a forgetful agent and all 3 memory types, and compare.

In [14]:
# ── Side-by-side comparison — modern approach ──────────────
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

CONVERSATION = [
    "My name is Ravi and I am from Hyderabad.",
    "I work as a Data Scientist.",
    "My favourite framework is LangChain.",
    "What is my name?",
    "What city am I from?",
    "What do I do for work?",
]

# ── Helper: run a conversation and show results ─────────────
def run_with_history(name, chain_fn):
    """Run CONVERSATION through a chain function and print results."""
    print(f"\n{'='*55}")
    print(f"  {name}")
    print(f"{'='*55}")
    for i, msg in enumerate(CONVERSATION, 1):
        response = chain_fn(msg)
        status = "✅" if any(w in response.lower() for w in ["ravi","hyderabad","data","scientist","langchain"]) else "❌"
        print(f"  Turn {i} {status} | Q: {msg[:40]}")
        print(f"           | A: {response[:70]}..." if len(response)>70 else f"           | A: {response}")

# ── Agent 1: No memory ─────────────────────────────────────
no_mem_history = []
def forgetful(msg):
    return llm.invoke([HumanMessage(content=msg)]).content

run_with_history("❌  NO MEMORY (Forgetful)", forgetful)

# ── Agent 2: Buffer memory ─────────────────────────────────
buf_history = ChatMessageHistory()
sys_msg = SystemMessage(content="You are a helpful assistant. Be concise.")
def buffer_chat(msg):
    buf_history.add_user_message(msg)
    resp = llm.invoke([sys_msg] + buf_history.messages).content
    buf_history.add_ai_message(resp)
    return resp

run_with_history("✅  BUFFER MEMORY", buffer_chat)

# ── Agent 3: Window memory (k=4) ──────────────────────────
win_history = ChatMessageHistory()
def window_chat(msg, k=4):
    win_history.add_user_message(msg)
    recent = win_history.messages[-(k*2):]  # last k turns
    resp = llm.invoke([sys_msg] + recent).content
    win_history.add_ai_message(resp)
    return resp

run_with_history("✅  WINDOW MEMORY (k=4)", window_chat)

# ── Agent 4: Summary memory ────────────────────────────────
sum_mem = SummaryMemory(llm=llm, keep_last=4)  # defined in cell above
run_with_history("✅  SUMMARY MEMORY", sum_mem.chat)




  ❌  NO MEMORY (Forgetful)
  Turn 1 ✅ | Q: My name is Ravi and I am from Hyderabad.
           | A: Nice to meet you, Ravi. Hyderabad is a beautiful city with a rich hist...
  Turn 2 ✅ | Q: I work as a Data Scientist.
           | A: That's a fascinating field. As a Data Scientist, you must work with co...
  Turn 3 ✅ | Q: My favourite framework is LangChain.
           | A: LangChain is a popular open-source framework for building applications...
  Turn 4 ❌ | Q: What is my name?
           | A: I don't know your name. I'm a large language model, I don't have the a...
  Turn 5 ✅ | Q: What city am I from?
           | A: I don't have any information about your location. I'm a large language...
  Turn 6 ❌ | Q: What do I do for work?
           | A: I don't have any information about your work or personal life. I'm a l...

  ✅  BUFFER MEMORY
  Turn 1 ✅ | Q: My name is Ravi and I am from Hyderabad.
           | A: Hello Ravi, nice to meet you. How can I assist you today?
  Turn 2 ✅ | Q: I 

---
## 5. Guided Practice — Build a Switchable Memory Chatbot

The Day 3 deliverable: a chatbot where you can choose the memory type at startup.

Let's build it together.

In [16]:
# ── MemoryChatbot — modern implementation ──────────────────
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.3)
SYSTEM = SystemMessage(content="You are a helpful assistant. Be concise.")

class MemoryChatbot:

    def __init__(self, memory_type="buffer", window_k=5):
        self.memory_type = memory_type
        self.window_k    = window_k
        self.turn_count  = 0
        self._init_memory()
        print(f"✅ Chatbot created with '{memory_type}' memory.")

    def _init_memory(self):
        if self.memory_type == "buffer":
            self._history = ChatMessageHistory()
        elif self.memory_type == "window":
            self._history = WindowChatHistory(k=self.window_k)
        elif self.memory_type == "summary":
            self._summary_mem = SummaryMemory(llm=llm, keep_last=4)
        elif self.memory_type == "entity":
            self._entity_mem = EntityMemory(llm=llm)
        else:
            raise ValueError(f"Unknown memory type: {self.memory_type}")

    def chat(self, user_input):
        self.turn_count += 1
        if self.memory_type in ("buffer", "window"):
            self._history.add_user_message(user_input)
            response = llm.invoke([SYSTEM] + self._history.messages).content
            self._history.add_ai_message(response)
        elif self.memory_type == "summary":
            response = self._summary_mem.chat(user_input)
        elif self.memory_type == "entity":
            response = self._entity_mem.chat(user_input)
        return response

    def show_memory(self):
        print(f"\n--- Memory ({self.memory_type}) after {self.turn_count} turns ---")
        if self.memory_type in ("buffer", "window"):
            msgs = self._history.messages
            print(f"Messages stored: {len(msgs)}")
            for m in msgs:
                role = "Human" if isinstance(m, HumanMessage) else "AI"
                print(f"  {role}: {m.content[:80]}")
        elif self.memory_type == "summary":
            print(f"Summary: {self._summary_mem.summary[:300]}")
            print(f"Recent messages: {len(self._summary_mem.recent)}")
        elif self.memory_type == "entity":
            print("Entity store:")
            for k, v in self._entity_mem.entities.items():
                print(f"  📌 {k}: {v}")
        print()

    def reset(self):
        self._init_memory()
        self.turn_count = 0
        print("🔄 Memory cleared.")


# ── Test with buffer memory ────────────────────────────────
bot = MemoryChatbot(memory_type="buffer")
print()

print(bot.chat("Hi! I am Ravi, a Data Scientist from Hyderabad."))
print(bot.chat("I am working on a RAG project using LangChain and ChromaDB."))
print(bot.chat("What do you know about me and my project?"))

bot.show_memory()

✅ Chatbot created with 'buffer' memory.

Hi Ravi, nice to meet you. How can I assist you with data science today?
LangChain and ChromaDB are interesting tools. What specific challenges or questions do you have regarding your RAG (Retrieve, Augment, Generate) project?
From our conversation, I know that:
1. Your name is Ravi.
2. You are a Data Scientist.
3. You are from Hyderabad.
4. You are working on a RAG (Retrieve, Augment, Generate) project.
5. You are using LangChain and ChromaDB for your project.

That's all I know so far. If you'd like to share more, I'm here to help.

--- Memory (buffer) after 3 turns ---
Messages stored: 6
  Human: Hi! I am Ravi, a Data Scientist from Hyderabad.
  AI: Hi Ravi, nice to meet you. How can I assist you with data science today?
  Human: I am working on a RAG project using LangChain and ChromaDB.
  AI: LangChain and ChromaDB are interesting tools. What specific challenges or questi
  Human: What do you know about me and my project?
  AI: From our con

In [17]:
# Now try the same bot with summary memory — notice the memory format is different
summary_bot = MemoryChatbot(memory_type="summary")
print()

print(summary_bot.chat("Hi! I am Ravi, a Data Scientist from Hyderabad."))
print(summary_bot.chat("I am working on a RAG project using LangChain and ChromaDB."))
print(summary_bot.chat("What do you know about me and my project?"))

summary_bot.show_memory()

✅ Chatbot created with 'summary' memory.

Hi Ravi, nice to meet you. What can I help you with today?
That sounds interesting. RAG (Retrieval-Augmented Generation) is a powerful concept. How can I assist you with your LangChain and ChromaDB project? Do you have a specific question or issue?
  [Summary updated: 203 chars]
  [Summary updated: 216 chars]
I know that your name is Ravi, you're a Data Scientist from Hyderabad, and you're working on a RAG (Retrieval-Augmented Generation) project using LangChain and ChromaDB. That's all I know so far. If you'd like to share more, I'd be happy to learn more about your project.

--- Memory (summary) after 3 turns ---
Summary: We have just started our conversation, and I greeted Ravi. Ravi is a Data Scientist from Hyderabad, but no other information has been shared yet. The conversation is expected to continue from this introductory point.
Recent messages: 4



In [18]:
# Entity memory — watch how it builds a knowledge base of named things
entity_bot = MemoryChatbot(memory_type="entity")
print()

print(entity_bot.chat("I am Ravi, working at DataCorp."))
print(entity_bot.chat("My colleague Priya is the lead engineer on ProjectX."))
print(entity_bot.chat("ProjectX is a customer support agent built with CrewAI."))
print(entity_bot.chat("Tell me what you know about Priya and ProjectX."))

entity_bot.show_memory()

✅ Chatbot created with 'entity' memory.

You work at DataCorp. What do you need help with?
That's correct, Priya is indeed the lead engineer on ProjectX. Is there something specific you'd like to know or discuss about ProjectX or Priya's work?
ProjectX, led by Priya, is a customer support agent that utilizes CrewAI technology. What's your question about it?
I know that Priya is the lead engineer on ProjectX, and ProjectX is a customer support agent. That's the information I have so far.

--- Memory (entity) after 4 turns ---
Entity store:
  📌 Ravi: working at DataCorp
  📌 DataCorp: Ravi's workplace
  📌 Priya: lead engineer on ProjectX
  📌 ProjectX: has Priya as lead engineer. a customer support agent
  📌 CrewAI: used to build ProjectX



---
## 6. Token Cost Comparison — See the Difference in Numbers

Memory has a real cost. Let's measure how many tokens each memory type consumes after the same 5-turn conversation.

In [19]:
# ── Token cost comparison ───────────────────────────────────
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

TURNS = [
    "My name is Ravi. I work as a Data Scientist at DataCorp in Hyderabad.",
    "I have been using Python for 4 years and recently started learning LangChain.",
    "My current project involves building a RAG pipeline for internal document search.",
    "I am also exploring multi-agent systems with CrewAI for a chatbot use case.",
    "Based on all this, what should I focus on next in my learning journey?",
]

llm_cost = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
SYSTEM = SystemMessage(content="You are a helpful assistant. Be concise.")
results = {}

# ── Buffer ─────────────────────────────────────────────────
buf_h = ChatMessageHistory()
for turn in TURNS:
    buf_h.add_user_message(turn)
    resp = llm_cost.invoke([SYSTEM] + buf_h.messages).content
    buf_h.add_ai_message(resp)
results["Buffer"] = sum(len(m.content) for m in buf_h.messages)

# ── Window (k=3) ───────────────────────────────────────────
win_h = WindowChatHistory(k=3)
for turn in TURNS:
    win_h.add_user_message(turn)
    resp = llm_cost.invoke([SYSTEM] + win_h.messages).content
    win_h.add_ai_message(resp)
# Measure total stored (not just what's visible)
all_win = win_h._messages if hasattr(win_h, '_messages') else []
results["Window (k=3)"] = sum(len(m.content) for m in win_h.messages)

# ── Summary ────────────────────────────────────────────────
sum_cost = SummaryMemory(llm=llm_cost, keep_last=4)
for turn in TURNS:
    sum_cost.chat(turn)
summary_size = len(sum_cost.summary) + sum(len(m.content) for m in sum_cost.recent)
results["Summary"] = summary_size

# ── Print results ──────────────────────────────────────────
print("=== Memory Size After 5 Turns ===")
print()
for mem_name, size in results.items():
    print(f"  {mem_name:20s} → {size:5d} chars sent to LLM each turn")

print()
smallest = min(results.values())
for name, size in results.items():
    ratio = size / smallest
    bar = "█" * int(ratio * 10)
    print(f"  {name:20s} {bar} ({ratio:.1f}x)")

print()
print("📊 Smaller = fewer tokens sent = lower cost per turn.")



  [Summary updated: 256 chars]
  [Summary updated: 301 chars]
  [Summary updated: 270 chars]
  [Summary updated: 313 chars]
  [Summary updated: 413 chars]
  [Summary updated: 382 chars]
=== Memory Size After 5 Turns ===

  Buffer               →  1430 chars sent to LLM each turn
  Window (k=3)         →  1001 chars sent to LLM each turn
  Summary              →  1042 chars sent to LLM each turn

  Buffer               ██████████████ (1.4x)
  Window (k=3)         ██████████ (1.0x)
  Summary              ██████████ (1.0x)

📊 Smaller = fewer tokens sent = lower cost per turn.


---
## 7. Extend the AgentLLM Class — Add Memory Support

In Day 1, we built the `AgentLLM` wrapper class. Now let's extend it to support memory.

This is exactly how production AI systems are built — a single wrapper that hides both the provider AND the memory type.

In [21]:
# ── Extended AgentLLM class — Day 3 update ─────────────────
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_community.chat_message_histories import ChatMessageHistory

class AgentLLM:
    """
    Reusable LLM wrapper for the Agentic AI course.
    Day 1: Multi-provider support (Groq, Gemini)
    Day 3: Memory support added (buffer, window, summary, entity)

    Usage:
        agent = AgentLLM(provider="groq", memory_type="buffer")
        agent.ask("Hello")            # one-shot, no memory
        agent.chat("My name is Ravi") # memory-enabled conversation
        agent.show_memory()           # inspect what's stored
        agent.reset_memory()          # clear and start fresh
    """

    def __init__(self, provider="groq", model=None, temperature=0,
                 max_tokens=500, memory_type="buffer", window_k=5):
        self.provider    = provider
        self.memory_type = memory_type
        self.window_k    = window_k
        self._system     = SystemMessage(content="You are a helpful assistant. Be concise.")

        if provider == "groq":
            self.llm = ChatGroq(
                model=model or "llama-3.3-70b-versatile",
                temperature=temperature,
                max_tokens=max_tokens,
            )
        elif provider == "gemini":
            from langchain_google_genai import ChatGoogleGenerativeAI
            self.llm = ChatGoogleGenerativeAI(
                model=model or "gemini-2.5-flash-lite",
                temperature=temperature,
                max_tokens=max_tokens,
            )
        else:
            raise ValueError(f"Unknown provider: {provider}. Use 'groq' or 'gemini'.")

        self._init_memory()
        print(f"✅ AgentLLM ready | provider={provider} | memory={memory_type}")

    def _init_memory(self):
        if self.memory_type == "none":
            self._memory = None
        elif self.memory_type == "buffer":
            self._memory = ChatMessageHistory()
        elif self.memory_type == "window":
            self._memory = WindowChatHistory(k=self.window_k)
        elif self.memory_type == "summary":
            self._memory = SummaryMemory(llm=self.llm, keep_last=4)
        elif self.memory_type == "entity":
            self._memory = EntityMemory(llm=self.llm)
        else:
            raise ValueError(f"Unknown memory type: {self.memory_type}")

    def ask(self, question, system_prompt=None):
        # One-shot question — no memory used
        msgs = []
        if system_prompt:
            msgs.append(SystemMessage(content=system_prompt))
        msgs.append(HumanMessage(content=question))
        return self.llm.invoke(msgs).content

    def chat(self, message):
        # Memory-enabled conversation
        if self.memory_type == "none":
            raise ValueError("Memory disabled. Use ask() instead.")
        if self.memory_type in ("buffer", "window"):
            self._memory.add_user_message(message)
            response = self.llm.invoke([self._system] + self._memory.messages).content
            self._memory.add_ai_message(response)
            return response
        elif self.memory_type == "summary":
            return self._memory.chat(message)
        elif self.memory_type == "entity":
            return self._memory.chat(message)

    def stream(self, question, system_prompt=None):
        # Stream response token by token
        msgs = []
        if system_prompt:
            msgs.append(SystemMessage(content=system_prompt))
        msgs.append(HumanMessage(content=question))
        for chunk in self.llm.stream(msgs):
            print(chunk.content, end="", flush=True)
        print()

    def show_memory(self):
        # Inspect what the memory currently holds
        if not self._memory:
            print("Memory is disabled.")
            return
        print(f"\n--- Memory ({self.memory_type}) ---")
        if self.memory_type in ("buffer", "window"):
            for m in self._memory.messages:
                role = "Human" if isinstance(m, HumanMessage) else "AI"
                print(f"  {role}: {m.content[:80]}")
        elif self.memory_type == "summary":
            print(f"Summary: {self._memory.summary}")
            print(f"Recent turns: {len(self._memory.recent)}")
        elif self.memory_type == "entity":
            for k, v in self._memory.entities.items():
                print(f"  📌 {k}: {v}")
        print()

    def reset_memory(self):
        # Clear all memory and start fresh
        self._init_memory()
        print("🔄 Memory cleared.")


# ── Test ────────────────────────────────────────────────────
agent = AgentLLM(provider="groq", memory_type="summary")
print()

print(agent.chat("I am Ravi, a Data Scientist from Hyderabad."))
print(agent.chat("I am currently building a RAG pipeline for my company."))
print(agent.chat("Can you summarize what you know about me?"))

agent.show_memory()

✅ AgentLLM ready | provider=groq | memory=summary

Hello Ravi, nice to meet you. How can I assist you today?
A Retrieval-Augmented Generation (RAG) pipeline can be complex. What specific challenges or areas of the pipeline are you struggling with or would like help with?
  [Summary updated: 245 chars]
  [Summary updated: 260 chars]
You're Ravi, a Data Scientist from Hyderabad, and you're currently building a Retrieval-Augmented Generation (RAG) pipeline for your company. That's the extent of what I know about you so far.

--- Memory (summary) ---
Summary: We have just started our conversation, and I greeted Ravi, a Data Scientist from Hyderabad. Ravi has introduced himself, but no other information has been shared yet. The conversation is expected to continue with Ravi sharing more details or asking a question.
Recent turns: 4



---
## ✅ Quick Check — Switch Memory Type in One Word

The Strategy Pattern in action — same code, different memory, just change one parameter:

In [22]:
# Same conversation, 4 different agents, one line change each

TEST_MSGS = [
    "My name is Ravi and I am building AI agents.",
    "I prefer using Groq over OpenAI because it's faster and free.",
    "What do you know about my preferences?",
]

for mem_type in ["buffer", "window", "summary", "entity"]:
    print(f"\n{'─'*50}")
    agent = AgentLLM(provider="groq", memory_type=mem_type, window_k=3)
    for msg in TEST_MSGS:
        response = agent.chat(msg)
    # Only show the final answer to the preference question
    print(f"Final answer ({mem_type}): {response[:120]}..." if len(response) > 120 else f"Final answer ({mem_type}): {response}")


──────────────────────────────────────────────────
✅ AgentLLM ready | provider=groq | memory=buffer
Final answer (buffer): From our conversation, I know you prefer using Groq over OpenAI, likely due to its speed and free access. You're also in...

──────────────────────────────────────────────────
✅ AgentLLM ready | provider=groq | memory=window
Final answer (window): From our conversation, I know you prefer using Groq over OpenAI because it's faster and free. That's the only preference...

──────────────────────────────────────────────────
✅ AgentLLM ready | provider=groq | memory=summary
  [Summary updated: 275 chars]
  [Summary updated: 238 chars]
Final answer (summary): You mentioned earlier that you prefer using Groq over OpenAI because it's faster and free, indicating that speed and cos...

──────────────────────────────────────────────────
✅ AgentLLM ready | provider=groq | memory=entity
Final answer (entity): From our conversation, I know you prefer using Groq over OpenAI for 

---
## 8. 🏆 Challenge — Build an Interactive Memory Chatbot

**Your turn (15 minutes):**

Build a chatbot that:
1. Lets the user choose the memory type at startup
2. Runs a real back-and-forth conversation in the terminal
3. Has a `/memory` command to inspect current memory
4. Has a `/reset` command to clear memory and start fresh
5. Has a `/quit` command to exit

Skeleton is below — fill in the missing parts:

In [23]:
def interactive_chatbot():
    """Interactive terminal chatbot with switchable memory."""

    print("\n" + "="*50)
    print("  🧠  Memory Chatbot")
    print("="*50)
    print("Commands: /memory  /reset  /quit")
    print()

    # Step 1: Ask user which memory type to use
    print("Choose memory type:")
    print("  1. buffer   — remembers everything")
    print("  2. window   — remembers last 5 turns")
    print("  3. summary  — compresses old history")
    print("  4. entity   — tracks named entities")
    print()

    choice = input("Enter memory type (buffer/window/summary/entity): ").strip().lower()
    if choice not in ["buffer", "window", "summary", "entity"]:
        choice = "buffer"
        print("Invalid choice, defaulting to 'buffer'.")

    # Step 2: Create the agent with chosen memory type
    agent = AgentLLM(provider="groq", memory_type=choice, temperature=0.3)
    print(f"\nChatbot ready! Using '{choice}' memory.")
    print("Type your message or a command.")
    print()

    # Step 3: Main conversation loop
    while True:
        user_input = input("You: ").strip()

        if not user_input:
            continue

        # Handle commands
        if user_input == "/quit":
            print("Goodbye!")
            break
        elif user_input == "/memory":
            agent.show_memory()
            continue
        elif user_input == "/reset":
            agent.reset_memory()
            print("Memory cleared! Starting fresh.\n")
            continue

        # TODO: Add your own command here — e.g. /switch to change memory type mid-conversation

        # Step 4: Get response from agent
        response = agent.chat(user_input)
        print(f"\nAI: {response}")
        print()


# Uncomment to run interactively:
# interactive_chatbot()

print("✅ interactive_chatbot() defined — uncomment the last line to run it!")
print("   It runs in the terminal — type messages and test all 4 memory types.")

✅ interactive_chatbot() defined — uncomment the last line to run it!
   It runs in the terminal — type messages and test all 4 memory types.


### 🎯 Your Turn — Extend the Chatbot

**Challenge extensions (pick any 2):**

1. Add a `/switch` command that lets the user change memory type mid-conversation
2. Add a `/tokens` command that estimates and shows how many tokens are in memory
3. Add a `/export` command that saves the current conversation to a `.txt` file
4. Add a `/persona` command that changes the system prompt (e.g., `"You are a Python tutor"`)
5. Build a **dual-bot** — two agents with different memory types running in parallel, answering the same question

In [24]:
# YOUR CODE HERE — extend the chatbot with at least 2 new features

# Example skeleton for /tokens command:
# import tiktoken
# encoder = tiktoken.get_encoding("cl100k_base")
# token_count = len(encoder.encode(agent.memory.buffer))
# print(f"Estimated tokens in memory: {token_count}")

---
## 9. Day 3 Wrap-up

### What you built today:
- ✅ Proved LLMs are stateless — zero memory without engineering
- ✅ Built buffer memory from scratch with Python dicts
- ✅ Built sliding window memory from scratch
- ✅ Used all 4 LangChain memory classes (Buffer, Window, Summary, Entity)
- ✅ Compared them side by side — same conversation, different results
- ✅ Measured token cost differences across memory types
- ✅ Extended `AgentLLM` with memory support (the class grows across all 23 sessions!)
- ✅ Built a switchable-memory chatbot

### The Key Insight:
> Memory is NOT an LLM feature. It is an **application-layer engineering decision** that you control completely. ChatGPT, Claude, Gemini — they're all doing exactly what you did today, just at billion-user scale.

### Memory Type Cheat Sheet:

| Type | Tokens | Best For |
|---|---|---|
| `ConversationBufferMemory` | Grows forever | Short demos, testing |
| `ConversationBufferWindowMemory` | Fixed (last K turns) | Long chats, cost control |
| `ConversationSummaryMemory` | Compact summary | Multi-session assistants |
| `ConversationEntityMemory` | Facts only | Personal assistants |

### Tomorrow — Day 4: LangChain Framework Deep Dive
We go deeper into LangChain's full architecture — Models, Prompts, Chains, Agents, Output Parsers, and Router Chains. You'll build a LangChain agent that automatically routes between different tools based on the question type.

---
### 📚 Want to explore more?
- [LangChain Memory docs](https://python.langchain.com/docs/modules/memory/)
- [ConversationChain reference](https://python.langchain.com/api_reference/langchain/chains/langchain.chains.conversation.base.ConversationChain.html)
- [ChatGPT Memory feature (OpenAI)](https://openai.com/index/memory-and-new-controls-for-chatgpt/)